# Run your own large language model (LLM) locally with Ollama

Onlie chatbots are convenient, but your prompts and data leave your machine. 
With [Ollama](https://ollama.com) you can run open models on your own computer instead.


**We run everything on the CPUs here.** 

## 1. Install Ollama (terminal)

Open a terminal in VS Code (Terminal -> New Terminal) and execute the following commands.

1. Download Ollama into your home directory (it stays there when the workspace restarts) and delete the GPU libraries we do not need:
   ```bash
   mkdir -p ~/.local
   curl -fsSL https://ollama.com/download/ollama-linux-amd64.tar.zst | zstd -d | tar -xf - -C ~/.local
   rm -rf ~/.local/lib/ollama/cuda_* ~/.local/lib/ollama/vulkan
   ```
2. Make the `ollama` command available in all terminals:
   ```bash
   echo 'export PATH="$HOME/.local/bin:$PATH"' >> ~/.bashrc
   source ~/.bashrc
   ```
3. Start the Ollama server in the background on the CPU only (`CUDA_VISIBLE_DEVICES=-1` hides all GPUs):
   ```bash
   CUDA_VISIBLE_DEVICES=-1 nohup ollama serve > ~/ollama.log 2>&1 &
   ```
   **After a workspace restart, you only need to repeat this step.**

## 2. Chat with a small model (terminal)

1. Download the model [`gemma3:1b`](https://ollama.com/library/gemma3) (about 800 MB):
   ```bash
   ollama pull gemma3:1b
   ```
2. By default, Ollama uses all CPU cores it can find, which slows down everyone on the shared server. Create a variant of the model that uses 4 threads. A [Modelfile](https://docs.ollama.com/modelfile) describes a model and its parameters:
   ```bash
   printf 'FROM gemma3:1b\nPARAMETER num_thread 4\n' > ~/Modelfile
   ollama create gemma3-cpu -f ~/Modelfile
   ```
3. Chat with the model. `--verbose` prints timing statistics after each answer:
   ```bash
   ollama run --verbose gemma3-cpu
   ```
   Ask a few questions, then type `/bye` to exit. How many tokens per second (`eval rate`) does the model generate?
4. In a second terminal, check where the model runs. The `PROCESSOR` column should show `100% CPU`:
   ```bash
   ollama ps
   ```

## 3. Use the model from Python

The [`ollama` Python package](https://github.com/ollama/ollama-python) talks to the server you started. Run the cells below.

In [ ]:
import ollama

MODEL = "gemma3-cpu"
OPTIONS = {"num_thread": 4} 


def ask(prompt, model=MODEL):
    """Send a prompt to a local model, print the answer and the generation speed."""
    response = ollama.generate(model=model, prompt=prompt, options=OPTIONS)
    tokens_per_s = response.eval_count / (response.eval_duration / 1e9)
    print(response.response)
    print(f"\n--- {model}: {response.eval_count} tokens, {tokens_per_s:.1f} tokens/s")
    return response.response


print([m.model for m in ollama.list().models])

In [ ]:
answer = ask("Explain in two sentences what a polymer is.")

### Task: Ssolve the Python exercises with the model

Use `ask()` to solve the questions in [evaluate_python_skills.ipynb](evaluate_python_skills.ipynb) (e.g., the Unicode code point question).

- Copy the generated code into a new cell and run it. Does it work? Do the tests the model wrote actually pass?

## 5. Answer briefly:

1.  Would you use your own LLM for helping you with Python code, and why?
2. When would you prefer a local model over a cloud chatbot such as ChatGPT or Gemini (think of privacy, cost, reproducibility, and offline use)? When not?

## 6. Clean up

Models take up disk space in your home directory (`~/.ollama`). Delete the ones you no longer need and stop the server:
```bash
ollama list
ollama rm gemma3:1b
pkill ollama
```